# **APLICACIÓN DEL RANKING ELO EN FÚTBOL**

En este estudio se va a aplicar un modelo de *ranking Elo*. Este modelo es una forma objetiva de asignar una puntuación numérica (*rating*) a cada participante y actualizar dicha puntuación después de cada partido, basándose en el resultado real y en el resultado esperado.

Este modelo es útil para medir una valoración dinámica de cada equipo, teniendo en cuenta factores como la ventaja de jugar en casa, la fuerza del rival y la importancia de cada competición.

---

## **1. Fórmula del rating ELO**

La fórmula general del *rating Elo* es la siguiente:

$$
R_n = R_o + K \cdot (W - W_e)
$$

Donde:

- $R_n$ = nuevo rating,  
- $R_o$ = rating anterior,  
- $K$ = peso de la competición,  
- $W$ = resultado del partido,  
- $W_e$ = resultado esperado del partido. 

### **1.1. Peso del torneo ($K$)**

El valor de $K$ determina cuánto afecta un partido al rating. En este proyecto se utilizan los siguientes valores base según el tipo de competición:

- **50** para grandes torneos continentales,  
- **40** para las cinco grandes ligas europeas,  
- **30** para otras competiciones importantes,  
- **20** para competiciones de menor prioridad.  

El valor de $K$ se ajusta posteriormente según la **diferencia de goles**:

- **Victoria por 1 gol** → sin ajuste.  
- **Victoria por 2 goles** → $1.5 \cdot K$  
- **Victoria por 3 goles** → $1.75 \cdot K$  
- **Victoria por 4 o más goles** (donde $N$ es la diferencia de goles) →  

$$
K \cdot \left(1.75 + \frac{N - 3}{8}\right)
$$

### **1.2. Resultado real del partido ($W$)**

El resultado real del partido se representa como:

- **1.0** para victoria,  
- **0.5** para empate,  
- **0.0** para derrota.  

### **1.3. Resultado esperado del partido ($W_e$)**

El resultado esperado se calcula mediante la fórmula de expectativa Elo:

$$
W_e = \frac{1}{10^{-dr/400} + 1}
$$

Donde:

- $dr$ es la diferencia de rating entre ambos equipos, y  
- se añade una **bonificación de +100 puntos** al equipo local para reflejar la ventaja de jugar en casa.

---

## **2. Preparación de los datos**

Preparamos los datos iniciales que necesitaremos. Para los clubes, vamos a usar como Elo inicial los valores medianos de las ligas del país al final de la temporada 20-21. Extraídos de [ClubElo](http://clubelo.com/), [EloRatings](https://www.eloratings.net/), [FiveThirtyEight](https://data.fivethirtyeight.com/), o los coeficientes UEFA.

In [54]:
import os
import pandas as pd
import json
from typing import Tuple

# Estructura de carpetas
DATA_PATH = r"D:\data"
RAW_DATA_PATH = os.path.join(DATA_PATH, "raw")
SILVER_DATA_PATH = os.path.join(DATA_PATH, "silver")

# Lectura de los dataframes necesarios
match_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_1", "match_clean_1.csv"), sep=";")
team_df = pd.read_csv(os.path.join(SILVER_DATA_PATH, "cleaning_1", "team_clean_1.csv"), sep=";")

# Transformación de matches
matches_data = match_df.copy()

# Añadimos el factor K por competición
tournament_k = {"la_liga": 40, "premier_league": 40, "bundesliga": 40, "serie_a": 40, "ligue_1": 40, "champions_league": 50, "europa_league": 30, "copa_del_rey": 30, "la_liga_hypermotion": 20,
                "championship": 20, "eredivise": 30, "liga_portugal": 30, "conference_league": 30, "copa_libertadores": 20, "liga_mx": 20, "serie_a_brazil": 20, "liga_profesional": 20,
                "first_division_a": 20, "major_league_soccer": 20, "saudi_pro_league": 20, "super_lig": 20, "superligaen": 20, "swiss_super_league": 20, "allvenskan": 20, "eliteserien": 20,
                "conmebol_sudamericana": 20}
matches_data["TournamentWeight"] = matches_data["League"].map(tournament_k)

# Convertios fecha
matches_data["Date"] = pd.to_datetime(matches_data["Date"], format="%d/%m/%Y")
matches_data = matches_data.sort_values(by=["Date", "ID"]).reset_index(drop=True)

# Lectura del diccionario con los coeficientes del final de temporada para cada país
with open(os.path.join(os.getcwd(), "utils", "dict_elo.json"), "r", encoding="utf-8") as f:
    dict_elo_country = json.load(f)

# Aplicamos a los equipos según su país
team_elo_df = team_df.copy()
team_elo_df["EloRating"] = team_elo_df["Country"].map(dict_elo_country)

# Sacamos los que no tienen país ni elo rating
team_elo_df = team_elo_df.dropna(subset=["Country", "EloRating"])
team_elo_dict = dict(zip(team_elo_df["ID"], team_elo_df["EloRating"]))

---

## **3. Creación del algoritmo**

El algoritmo itera por todos los partidos que tenemos disponibles y busca un nuevo Elo rating según los valores anteriores. Observamos también los mejores 20 equipos, y podemos decir que cuadra según su rendimiento actual.

In [60]:
# Añadimos home elo y away elo al dataframe
matches_data["HomeElo"] = None
matches_data["AwayElo"] = None

# Para cada partido según nuestro orden, vamos a calcular el ELO de los dos equipos
for index, row in matches_data.iterrows():

    # Información que necesitaremos
    home_team = row["HomeTeam"]
    away_team = row["AwayTeam"]
    home_elo = team_elo_dict.get(home_team, 1000)
    away_elo = team_elo_dict.get(away_team, 1000)
    home_score = row["HomeScore"]
    away_score = row["AwayScore"]
    k = row["TournamentWeight"]

    # Calculamos la diferencia de goles para K - Home
    home_diff_goals = home_score - away_score
    if home_diff_goals <= 1:
        home_k = k
    elif home_diff_goals == 2:
        home_k = 1.5*k
    elif home_diff_goals == 3:
        home_k = 1.75*k
    else:
        home_k = k*(1.75 + (home_diff_goals - 3)/8)

    # Calculamos la diferencia de goles para K - Away
    away_diff_goals = away_score - home_score
    if away_diff_goals <= 1:
        away_k = k
    elif away_diff_goals == 2:
        away_k = 1.5*k
    elif away_diff_goals == 3:
        away_k = 1.75*k
    else:
        away_k = k*(1.75 + (away_diff_goals - 3)/8)

    # Resultado del partido
    home_w = 1 if home_score > away_score else 0.5 if home_score == away_score else 0
    away_w = 1 if home_score < away_score else 0.5 if home_score == away_score else 0

    # Diferencia de Elo por equipo
    home_dr = (home_elo - away_elo) + 100  
    away_dr = (away_elo - home_elo)

    # Resultado esperado por equipo
    home_we = 1 / (1 + 10**(-home_dr/400))
    away_we = 1 / (1 + 10**(-away_dr/400))

    # Nuevos Elo
    home_new_elo = home_elo + home_k * (home_w - home_we)
    away_new_elo = away_elo + away_k * (away_w - away_we)

    # Añadimos al diccionario
    team_elo_dict[home_team] = home_new_elo
    team_elo_dict[away_team] = away_new_elo

    # Añadimos al dataframe
    matches_data.at[index, "HomeElo"] = home_new_elo
    matches_data.at[index, "AwayElo"] = away_new_elo

In [61]:
# Aplicamos y buscamos los mejores 10 equipos
team_df["EloRating"] = team_df["ID"].map(team_elo_dict)
best_10_teams = team_df.sort_values(by="EloRating", ascending=False).head(10).reset_index(drop=True)

# Print
for index, row in best_10_teams.iterrows():
    print(f"{index+1}. {row["Name"]} ({row["EloRating"]})")

1. FC Bayern München (1993.0339158338093)
2. Arsenal (1961.3417275696402)
3. Barcelona (1939.423913123183)
4. Paris Saint-Germain (1910.1681850875827)
5. Real Madrid (1878.563291474201)
6. Manchester United (1869.0466853989894)
7. Manchester City (1861.1189634490756)
8. Bournemouth (1805.8156826222914)
9. Inter (1801.1954733328903)
10. Aston Villa (1791.682551442376)


---